# Notebook 1: BERT vs GPT - Demos Prácticos

**Sesión 4:** BERT vs GPT y Uso Responsable  
**Profesor:** Francisco Pérez Carbajal  
**Duración:** ~20 minutos (2 demos)  

---

## 📚 Contenido

### Parte 1: BERT - Text Classification (~10 min)
- Cargar modelo BERT pre-entrenado
- Tokenización
- Clasificación de sentiment
- Ejercicio práctico

### Parte 2: GPT - Text Generation (~10 min)
- Cargar modelo GPT-2
- Generación desde prompts
- Experimentar con temperature
- Ejercicio creativo

---

## 🎯 Objetivos

- ✅ Ver BERT en acción (encoder-only)
- ✅ Ver GPT en acción (decoder-only)
- ✅ Entender diferencias prácticas
- ✅ Experimentar con ambos modelos

---

# Parte 1: BERT - Text Classification

## ¿Qué es BERT?

- **Bidirectional Encoder Representations from Transformers**
- Arquitectura: **Encoder-only** (solo la parte que entiende)
- Objetivo: **Representar y entender** texto
- Uso común: Classification, NER, Question Answering

## Demo: Sentiment Analysis

Vamos a clasificar reseñas como positivas o negativas.

## Setup e Instalación

In [ ]:
# Instalar dependencias
!pip install transformers torch -q

In [ ]:
# Imports necesarios
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports completados")
print(f"✓ PyTorch version: {torch.__version__}")

## Método 1: Pipeline (Fácil y Rápido)

Hugging Face proporciona pipelines que hacen todo el trabajo por nosotros.

In [ ]:
# Crear pipeline de sentiment analysis
# Esto descarga un modelo BERT pre-entrenado y fine-tuned para sentiment

print("Cargando modelo BERT...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✓ Modelo cargado")

In [ ]:
# Probar con ejemplos
ejemplos = [
    "This movie is absolutely amazing! I loved every minute.",
    "Terrible experience. Worst film I've ever seen.",
    "It was okay, nothing special but not terrible either.",
    "The acting was great but the plot was confusing."
]

print("Clasificando sentimientos:\n")
for texto in ejemplos:
    resultado = sentiment_analyzer(texto)[0]
    label = resultado['label']
    score = resultado['score']
    emoji = "😊" if label == "POSITIVE" else "😞"
    
    print(f"{emoji} [{label}] (confianza: {score:.2f})")
    print(f"   Texto: {texto[:70]}...")
    print()

## Método 2: Manual (Entender el Proceso)

Veamos qué hace BERT "por dentro":

In [ ]:
# Cargar tokenizer y modelo manualmente
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print("Cargando tokenizer y modelo...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)
print("✓ Cargado")

In [ ]:
# Paso 1: Tokenización
texto = "This movie is fantastic!"

print(f"Texto original: {texto}\n")

# Tokenizar
inputs = tokenizer(texto, return_tensors="pt", padding=True, truncation=True)

print("Tokens (IDs numéricos):")
print(inputs['input_ids'])
print("\nTokens (palabras):")
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

In [ ]:
# Paso 2: Pasar por BERT
with torch.no_grad():
    outputs = model(**inputs)

# Paso 3: Obtener predicción
logits = outputs.logits
prediccion = torch.argmax(logits, dim=1).item()
probabilidades = torch.softmax(logits, dim=1)[0]

labels = ["NEGATIVE", "POSITIVE"]

print(f"\nLogits (valores crudos): {logits[0].tolist()}")
print(f"\nProbabilidades:")
print(f"  NEGATIVE: {probabilidades[0]:.4f}")
print(f"  POSITIVE: {probabilidades[1]:.4f}")
print(f"\n✓ Predicción final: {labels[prediccion]}")

## 🎯 Tu Turno: Experimenta con BERT

In [ ]:
# Prueba con tus propias frases
tus_ejemplos = [
    "Add your own sentence here",
    "Try different sentiments",
    "What happens with neutral text?",
    # Agrega más aquí
]

print("Tus resultados:\n")
for texto in tus_ejemplos:
    if texto and len(texto.strip()) > 0:  # Skip empty
        resultado = sentiment_analyzer(texto)[0]
        print(f"{resultado['label']}: {texto}")
        print(f"  Confianza: {resultado['score']:.2f}\n")

## 💡 Observaciones sobre BERT

**Ventajas:**
- ✅ Excelente para **entender** y **clasificar** texto
- ✅ Ve el contexto completo (**bidireccional**)
- ✅ Modelos pre-entrenados disponibles
- ✅ Fine-tuning relativamente rápido

**Limitaciones:**
- ❌ **NO genera** texto nuevo
- ❌ Solo produce embeddings o clasificaciones
- ❌ Necesita fine-tuning para cada tarea

**¿Cuándo usar BERT?**
- Clasificación de texto
- Named Entity Recognition (NER)
- Question Answering
- Semantic search
- Cualquier tarea donde necesites **entender** texto

---

# Parte 2: GPT - Text Generation

## ¿Qué es GPT?

- **Generative Pre-trained Transformer**
- Arquitectura: **Decoder-only** (solo la parte que genera)
- Objetivo: **Generar** texto
- Uso común: Chatbots, creative writing, code generation

## Demo: Text Generation

Vamos a generar texto automáticamente desde prompts.

In [ ]:
# Imports para GPT
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("✓ Imports para GPT listos")

In [ ]:
# Cargar GPT-2 (versión pequeña)
print("Cargando GPT-2...")
gpt_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")

# Configurar pad token (GPT-2 no lo tiene por defecto)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

print("✓ GPT-2 cargado")
print(f"✓ Tamaño del modelo: ~124M parámetros")

## Generación Básica

In [ ]:
# Función helper para generar texto
def generar_texto(prompt, max_length=50, temperature=0.7, num_return=1):
    """
    Genera texto usando GPT-2
    
    Args:
        prompt: Texto inicial
        max_length: Longitud máxima del output
        temperature: Creatividad (0.1=conservador, 1.0=creativo)
        num_return: Número de variaciones
    """
    # Tokenizar
    inputs = gpt_tokenizer.encode(prompt, return_tensors="pt")
    
    # Generar
    outputs = gpt_model.generate(
        inputs,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=num_return,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=gpt_tokenizer.eos_token_id
    )
    
    # Decodificar
    resultados = []
    for i, output in enumerate(outputs):
        texto = gpt_tokenizer.decode(output, skip_special_tokens=True)
        resultados.append(texto)
    
    return resultados

print("✓ Función de generación lista")

In [ ]:
# Ejemplo 1: Completar una historia
prompt = "Once upon a time, in a land far away, there lived a"

print(f"Prompt: {prompt}\n")
print("Generando...\n")

resultados = generar_texto(prompt, max_length=60, temperature=0.7, num_return=3)

for i, texto in enumerate(resultados, 1):
    print(f"Variación {i}:")
    print(texto)
    print("-" * 80)
    print()

## Experimentar con Temperature

**Temperature** controla la creatividad del modelo:
- 🧊 **Baja (0.1-0.3):** Conservador, predecible, coherente
- 🌡️ **Media (0.7-0.9):** Balance creatividad/coherencia
- 🔥 **Alta (1.0-1.5):** Muy creativo, menos coherente

In [ ]:
# Comparar diferentes temperatures
prompt = "The future of artificial intelligence is"

temperatures = [0.3, 0.7, 1.2]

print(f"Prompt: {prompt}\n")
print("=" * 80)

for temp in temperatures:
    print(f"\n🌡️ Temperature = {temp}")
    print("-" * 80)
    
    resultado = generar_texto(prompt, max_length=50, temperature=temp, num_return=1)
    print(resultado[0])
    print()

## Diferentes Tipos de Prompts

In [ ]:
# Ejemplo 2: Pregunta y respuesta
prompt = "Q: What is machine learning?\nA:"

print("Prompt:")
print(prompt)
print("\nRespuesta GPT-2:")
print(generar_texto(prompt, max_length=80, temperature=0.5)[0])

In [ ]:
# Ejemplo 3: Creative writing
prompt = "Write a haiku about neural networks:"

print("Prompt:")
print(prompt)
print("\nRespuesta GPT-2:")
print(generar_texto(prompt, max_length=40, temperature=0.8)[0])

In [ ]:
# Ejemplo 4: Código (GPT-2 no es experto, pero intenta)
prompt = "# Python function to calculate fibonacci\ndef fibonacci(n):"

print("Prompt:")
print(prompt)
print("\nRespuesta GPT-2:")
print(generar_texto(prompt, max_length=100, temperature=0.3)[0])

## 🎯 Tu Turno: Experimenta con GPT

In [ ]:
# Tu propio prompt
tu_prompt = "Write your prompt here..."

# Ajusta estos parámetros
tu_max_length = 60
tu_temperature = 0.7
tu_num_return = 2

if tu_prompt and len(tu_prompt.strip()) > 5:
    print(f"Tu prompt: {tu_prompt}\n")
    resultados = generar_texto(
        tu_prompt,
        max_length=tu_max_length,
        temperature=tu_temperature,
        num_return=tu_num_return
    )
    
    for i, texto in enumerate(resultados, 1):
        print(f"\nGeneración {i}:")
        print(texto)
        print("-" * 60)
else:
    print("✏️ Escribe tu prompt arriba y ejecuta de nuevo")

## 💡 Observaciones sobre GPT

**Ventajas:**
- ✅ **Genera** texto nuevo y coherente
- ✅ Versatilidad (muchas tareas con prompts)
- ✅ No necesita fine-tuning para tareas básicas
- ✅ In-context learning (ejemplos en el prompt)

**Limitaciones:**
- ❌ **Unidireccional** (solo ve tokens anteriores)
- ❌ Puede generar contenido falso ("hallucinations")
- ❌ GPT-2 es limitado (GPT-3/4 son mucho mejores)
- ❌ Sesgos del training data

**¿Cuándo usar GPT?**
- Text generation
- Chatbots
- Creative writing
- Code completion
- Cualquier tarea donde necesites **generar** texto

---

# Comparación Final: BERT vs GPT

## Tabla Resumen

| Característica | BERT | GPT |
|----------------|------|-----|
| **Arquitectura** | Encoder-only | Decoder-only |
| **Objetivo** | Entender/Representar | Generar |
| **Dirección** | Bidireccional | Unidireccional |
| **Training** | Masked LM | Next token prediction |
| **Output** | Embeddings/clasificación | Texto nuevo |
| **Fine-tuning** | Necesario por tarea | Opcional (prompt basta) |
| **Uso común** | Classification, NER, QA | Generation, chatbots |

## ¿Cuál usar?

**Usa BERT cuando necesites:**
- 🔍 Clasificar texto
- 🏷️ Extraer entidades (NER)
- ❓ Responder preguntas (con contexto dado)
- 🔎 Búsqueda semántica
- 📊 Entender y analizar texto

**Usa GPT cuando necesites:**
- ✍️ Generar texto nuevo
- 💬 Crear chatbots
- 🎨 Creative writing
- 💻 Code generation
- 📝 Completar/continuar texto

## Híbridos y Variantes

**Encoder-Decoder completo:**
- T5, BART, FLAN-T5
- Bueno para traducción, summarization

**Modelos modernos:**
- GPT-3, GPT-4 (decoder-only más grandes)
- RoBERTa, ALBERT (BERT mejorados)
- Sentence-BERT (embeddings mejorados)

---

# Ejercicio Final: Combinar Ambos

Usa BERT para clasificar y GPT para responder.

In [ ]:
# Sistema simple que usa ambos modelos
def responder_con_contexto(user_input):
    """
    1. BERT clasifica el sentiment del input
    2. GPT genera respuesta apropiada al sentiment
    """
    # Paso 1: Clasificar sentiment con BERT
    sentiment = sentiment_analyzer(user_input)[0]
    label = sentiment['label']
    score = sentiment['score']
    
    print(f"📊 Sentiment detectado: {label} (confianza: {score:.2f})")
    
    # Paso 2: Crear prompt para GPT basado en sentiment
    if label == "POSITIVE":
        prompt = f"User: {user_input}\nAssistant: That's wonderful!"
    else:
        prompt = f"User: {user_input}\nAssistant: I'm sorry to hear that."
    
    # Paso 3: Generar respuesta con GPT
    respuesta = generar_texto(prompt, max_length=80, temperature=0.7, num_return=1)[0]
    
    # Extraer solo la respuesta del assistant
    respuesta_final = respuesta.split("Assistant:")[-1].strip()
    
    return respuesta_final

# Probar
ejemplos_usuario = [
    "I just got accepted to my dream university!",
    "My computer crashed and I lost all my work."
]

for user_msg in ejemplos_usuario:
    print(f"\n👤 Usuario: {user_msg}")
    respuesta = responder_con_contexto(user_msg)
    print(f"🤖 Assistant: {respuesta}")
    print("-" * 80)

---

# Resumen y Próximos Pasos

## ✅ Lo que hicimos hoy

### BERT (Encoder-only):
- ✓ Clasificación de sentiment
- ✓ Entendimos tokenización
- ✓ Vimos el proceso completo

### GPT (Decoder-only):
- ✓ Generación de texto
- ✓ Experimentamos con temperature
- ✓ Probamos diferentes prompts

### Comparación:
- ✓ BERT = Entender
- ✓ GPT = Generar
- ✓ Diferentes arquitecturas para diferentes tareas

## 🔮 Próximos temas

**En la siguiente parte de la clase:**
- Harms y sesgos en LLMs
- Performance disparities
- Social biases
- Uso responsable

**En próximas sesiones:**
- Fine-tuning de modelos
- LoRA y QLoRA (eficiente)
- Tu proyecto personal

---

## 📚 Referencias

**Papers:**
- BERT: Devlin+ 2018
- GPT-3: Brown+ 2020
- Harms: Bender & Gebru (2021), Weidinger et al. (2021)
- Biases: Abid et al. (2021), Schwartz et al. (2020)

**Tools:**
- Hugging Face Transformers
- Model Cards
- Bias testing kits

### Lecturas:
- Stanford CS324 Harms I lecture
- Foundation Models Report (Bommasani et al.)
- Stochastic Parrots paper

**¿Preguntas?** franciscop@ciencias.unam.mx